# Lab 2: Spark Structured Streaming with Google Cloud Pub/Sub (Avro)

In this lab, you will use PySpark Structured Streaming to read real-time Avro data from a Pub/Sub topic, process the orders to calculate their total value, and write the results back to a different Pub/Sub topic as JSON.

## 1. Setup Variables
Please update the `PROJECT_ID` below to match your Google Cloud project.

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"
INPUT_TOPIC = "orders-input"
OUTPUT_TOPIC = "orders-output"
CHECKPOINT_LOCATION = f"gs://{PROJECT_ID}-spark-checkpoints/lab02-notebook-avro"

## 2. Initialize SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_json, struct
from pyspark.sql.avro.functions import from_avro

spark = SparkSession.builder \
    .appName("Lab 2 - Pub/Sub Streaming Notebook") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession initialized.")

## 3. Read Stream from Pub/Sub

In [ ]:
raw_stream_df = spark.readStream \
    .format("pubsub") \
    .option("project", PROJECT_ID) \
    .option("topic", INPUT_TOPIC) \
    .load()

print(f"Created read stream from topic: {INPUT_TOPIC}")

## 4. Parse Avro Data
The incoming data is in binary format. We define the Avro schema and use `from_avro` to parse the payload.

In [ ]:
avro_schema = """
{
  "type": "record",
  "name": "Avro",
  "fields": [
    {"name": "orderId", "type": "string"},
    {"name": "amount", "type": "float"},
    {"name": "quantiy", "type": "int"}
  ]
}
"""

parsed_df = raw_stream_df \
    .select(from_avro(col("value"), avro_schema).alias("data")) \
    .select("data.*")

## 5. Process Data
Calculate the total value for each order (amount * quantity).

In [ ]:
processed_df = parsed_df.withColumn("total_value", col("amount") * col("quantiy"))

## 6. Write Stream to Pub/Sub
We serialize the processed result back to JSON and write it to the output topic.

In [ ]:
output_df = processed_df.select(
    to_json(struct(
        col("orderId"), 
        col("amount"), 
        col("quantiy"), 
        col("total_value")
    )).alias("value")
)

# Output must be binary for the pubsub connector
output_df = output_df.withColumn("value", col("value").cast("binary"))

query = output_df.writeStream \
    .format("pubsub") \
    .option("project", PROJECT_ID) \
    .option("topic", OUTPUT_TOPIC) \
    .option("checkpointLocation", CHECKPOINT_LOCATION) \
    .outputMode("append") \
    .start()

print(f"Streaming query started writing to {OUTPUT_TOPIC}...")
query.awaitTermination()